In [3]:
import pandas as pd
import polars as pl
import time
import requests
from pymongo import MongoClient
from pprint import pprint
import psycopg
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
import io



In [24]:
sql = """
WITH notes AS (
  SELECT
    serialnumber,
    STRING_AGG(remarks, ' | ' ORDER BY note_id)      AS remarks_all,
    STRING_AGG(opm_remarks, ' | ' ORDER BY note_id)  AS opm_remarks_all
  FROM d_sale_notes
  GROUP BY serialnumber
)
SELECT
  f.sale_id,
  f.serialnumber,
  f.listyear,
  f.daterecorded,
  f.assessedvalue,
  f.saleamount,
  f.salesratio,

  p.address,
  p.propertytype,
  p.residentialtype,
  p.latitude,
  p.longitude,

  t.town,
  n.nonusecode,

  notes.remarks_all,
  notes.opm_remarks_all
FROM f_sales AS f
JOIN d_property AS p
  ON p.property_id = f.property_id
JOIN d_town AS t
  ON t.town_id = p.town_id
LEFT JOIN d_non_use_code AS n
  ON n.nonusecode_id = f.nonusecode_id
LEFT JOIN notes
  ON notes.serialnumber = f.serialnumber;
"""
conn = psycopg.connect("host=127.0.0.1 port=5433 dbname=realestate user=postgres sslmode=disable")
cur = conn.cursor()

In [27]:
import polars as pl

cur.execute(sql)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]

df_ml = pl.DataFrame(rows, schema=cols)
print(df_ml.shape)
print(df_ml.head())

C:\Users\sebas\AppData\Local\Temp\ipykernel_29424\1936928366.py:7: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_ml = pl.DataFrame(rows, schema=cols)


(1141722, 16)
shape: (5, 16)
┌─────────┬────────────┬──────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ sale_id ┆ serialnumb ┆ listyear ┆ daterecord ┆ … ┆ town      ┆ nonusecod ┆ remarks_a ┆ opm_remar │
│ ---     ┆ er         ┆ ---      ┆ ed         ┆   ┆ ---       ┆ e         ┆ ll        ┆ ks_all    │
│ i64     ┆ ---        ┆ i64      ┆ ---        ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---       │
│         ┆ str        ┆          ┆ date       ┆   ┆           ┆ str       ┆ str       ┆ str       │
╞═════════╪════════════╪══════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 11      ┆ 40056      ┆ 2004     ┆ 2004-11-29 ┆ … ┆ RIDGEFIEL ┆ null      ┆ UNKNOWN   ┆ UNKNOWN   │
│         ┆            ┆          ┆            ┆   ┆ D         ┆           ┆           ┆           │
│ 27      ┆ 230181     ┆ 2023     ┆ 2024-02-20 ┆ … ┆ VERNON    ┆ null      ┆ UNKNOWN | ┆ RENOVATED │
│         ┆            ┆          ┆            ┆   ┆          

In [26]:
## Read CSV treating 'Sales Ratio' as string to avoid parse errors from thousand separators,
## then remove commas and cast to float.
#df = pl.read_csv(
#	r"C:\Users\sebas\OneDrive\Escritorio\Real_Estate_Sales_2001-2023_GL.csv",
#	infer_schema_length=10000,
#	schema_overrides={"Sales Ratio": pl.Utf8}
#)
#
#df = df.with_columns(
#	pl.col("Sales Ratio")
#	  .str.replace_all(",", "")
#	  .str.replace_all(" ", "")
#	  .cast(pl.Float64)
#	  .alias("Sales Ratio")
#)
#
#print(df.shape)
#print(df.head())
#print(df.tail())

In [32]:
numeric_types = {
	pl.Int8, pl.Int16, pl.Int32, pl.Int64,
	pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
	pl.Float32, pl.Float64}

num_cols  = [c for c, t in zip(df_ml.columns, df_ml.dtypes) if t in numeric_types]
cat_cols  = [c for c, t in zip(df_ml.columns, df_ml.dtypes) if t in (pl.Utf8, pl.Categorical)]
date_cosl = [c for c, t in zip(df_ml.columns, df_ml.dtypes) if t in (pl.Date, pl.Datetime)]


In [33]:
print("Num cols:", num_cols)
print("Cat cols:", cat_cols)
print("Date cols:", date_cols)

Num cols: ['sale_id', 'listyear', 'assessedvalue', 'saleamount', 'salesratio', 'latitude', 'longitude']
Cat cols: ['serialnumber', 'address', 'propertytype', 'residentialtype', 'town', 'nonusecode', 'remarks_all', 'opm_remarks_all']
Date cols: ['daterecorded']


In [34]:
df_ml.isnull().sum()

AttributeError: 'DataFrame' object has no attribute 'isnull'

## Numeric

In [30]:
df_ml.select(pl.col(num_cols)).describe()

statistic,sale_id,listyear,assessedvalue,saleamount,salesratio,latitude,longitude
str,f64,f64,f64,f64,f64,f64,f64
"""count""",1.141722e6,1.141722e6,1.141722e6,1.141722e6,1.141721e6,345184.0,345184.0
"""null_count""",0.0,0.0,0.0,0.0,1.0,796538.0,796538.0
"""mean""",570861.5,2011.673398,283327.51902,410450.975448,9.279262,41.498256,-72.87531
"""std""",329586.896357,7.018679,1.6561e6,5.0490e6,1766.534436,0.258617,0.441645
"""min""",1.0,2001.0,0.0,0.0,0.0,34.34581,-121.23091
"""25%""",285431.0,2005.0,89910.0,146100.0,0.4737,41.28944,-73.18851
"""50%""",570862.0,2012.0,141980.0,237500.0,0.604758,41.50214,-72.89923
"""75%""",856292.0,2018.0,230060.0,383750.0,0.764857,41.715321,-72.63055
"""max""",1.141722e6,2023.0,8.8151e8,5.0000e9,1.22642e6,44.93459,-71.18755
